In [ ]:
# scrape_quotes.py
import time
from typing import Iterator, Dict, List, Tuple
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

BASE = "https://quotes.toscrape.com"

def get_soup(url: str) -> BeautifulSoup:
    for attempt in range(3):
        r = requests.get(url, timeout=10, headers={"User-Agent": "quotes-mini-project/1.0"})
        if r.ok:
            return BeautifulSoup(r.text, "html.parser")
        time.sleep(1.5 * (attempt + 1))  # simple backoff
    r.raise_for_status()

def parse_list_page(soup: BeautifulSoup) -> Tuple[List[Dict], str | None]:
    out = []
    for q in soup.select(".quote"):
        text = q.select_one(".text").get_text(strip=True).strip("“”")
        author = q.select_one(".author").get_text(strip=True)
        tags = [t.get_text(strip=True) for t in q.select(".tag")]
        out.append({"text": text, "author": author, "tags": tags})
    next_link = soup.select_one("li.next > a")
    next_url = BASE + next_link["href"] if next_link else None
    return out, next_url

def iter_all_quotes(start_url: str = f"{BASE}/") -> Iterator[Dict]:
    url = start_url
    while url:
        soup = get_soup(url)
        items, url = parse_list_page(soup)
        for it in items:
            yield it
        time.sleep(0.5)  # be polite

if __name__ == "__main__":
    data = list(tqdm(iter_all_quotes(), desc="Scraping"))
    # quick preview
    print(f"Collected {len(data)} quotes. Sample:", data[0])